In [12]:
import pandas as pd
import pyfpgrowth

In [13]:
csv_in = 'df13_quiz_fp_r2.csv'
df = pd.read_csv(csv_in, sep=',', skiprows=0, header=0)
print(df.shape)
print(df.info())
display(df.head())

(3500, 2)
<class 'pandas.core.frame.DataFrame'>
Index: 3500 entries, 1 to 3500
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   Invoice    3500 non-null   object
 1   StockCode  3500 non-null   object
dtypes: object(2)
memory usage: 82.0+ KB
None


,Invoice,StockCode
1,X1088,m109
2,X1133,m115
3,X1170,m106
4,X1053,m101
5,X1181,m112


In [14]:
id2sc = sorted(list(set(df['StockCode'])))
sc2id = {}
for i in range(len(id2sc)):
    sc2id[id2sc[i]] = i

df['StockCode_ID'] = df['StockCode'].map(lambda x: sc2id[x])
display(df.head())

,Invoice,StockCode,StockCode_ID
1,X1088,m109,8
2,X1133,m115,14
3,X1170,m106,5
4,X1053,m101,0
5,X1181,m112,11


In [15]:
invoices = []
for r in df.groupby('Invoice'):
    s1 = set(r[1]['StockCode_ID'])
    invoices.append(list(s1))
print(f'Number of invoices: {len(invoices)}')

Number of invoices: 200


In [16]:
patterns = pyfpgrowth.find_frequent_patterns(invoices, 30)
print(f'Number of patterns: {len(patterns)}')

Number of patterns: 1524


In [17]:
rules = pyfpgrowth.generate_association_rules(patterns, 0.8)
print(f'Number of rules: {len(rules)}')

Number of rules: 29


In [18]:
results = []
for x in rules:
    ret = [x, rules[x][0], rules[x][1]]
    results.append(ret)
df_res = pd.DataFrame(results)
df_res.columns = ['LHS', 'RHS', 'Conf']
display(df_res.sort_values(by='Conf', ascending=False))

,LHS,RHS,Conf
0,"(2, 18, 19)","(17,)",0.864865
8,"(3, 5, 18)","(15,)",0.857143
7,"(2, 4, 10)","(17,)",0.853659
17,"(1, 14, 18)","(2,)",0.850000
14,"(1, 5, 16)","(17,)",0.833333
4,"(2, 3, 10)","(17,)",0.833333
25,"(2, 5, 18)","(17,)",0.825000
21,"(8, 12, 18)","(15,)",0.825000
13,"(9, 12, 15)","(18,)",0.822222
3,"(2, 6, 19)","(17,)",0.820513


In [19]:
n_all = len(invoices)
lift = []
for i in range(df_res.shape[0]):
    rhs = df_res.at[i, 'RHS']
    conf = df_res.at[i, 'Conf']
    n_rhs = 0
    for items in invoices:
        if set(items) >= set(rhs):
            n_rhs += 1
    lift1 = conf / (n_rhs / n_all)
    lift.append(lift1)
    
df_res['Lift'] = lift
display(df_res.sort_values(by='Conf', ascending=False))

,LHS,RHS,Conf,Lift
0,"(2, 18, 19)","(17,)",0.864865,1.310401
8,"(3, 5, 18)","(15,)",0.857143,1.328904
7,"(2, 4, 10)","(17,)",0.853659,1.293422
17,"(1, 14, 18)","(2,)",0.850000,1.393443
14,"(1, 5, 16)","(17,)",0.833333,1.262626
4,"(2, 3, 10)","(17,)",0.833333,1.262626
25,"(2, 5, 18)","(17,)",0.825000,1.250000
21,"(8, 12, 18)","(15,)",0.825000,1.279070
13,"(9, 12, 15)","(18,)",0.822222,1.405508
3,"(2, 6, 19)","(17,)",0.820513,1.243201


In [20]:
# 最も確信度が高いルール
max_conf_idx = df_res['Conf'].idxmax()
max_conf_rule = df_res.loc[max_conf_idx]

print(f"最も確信度が高いルール:")
print(f"LHS (ID): {max_conf_rule['LHS']}")
print(f"RHS (ID): {max_conf_rule['RHS']}")
print(f"Confidence: {max_conf_rule['Conf']:.4f}")
print(f"Lift: {max_conf_rule['Lift']:.4f}")

# StockCodeを確認
print(f"\nLHS (StockCode): {tuple([id2sc[i] for i in max_conf_rule['LHS']])}")
print(f"RHS (StockCode): {tuple([id2sc[i] for i in max_conf_rule['RHS']])}")

最も確信度が高いルール:
LHS (ID): (2, 18, 19)
RHS (ID): (17,)
Confidence: 0.8649
Lift: 1.3104

LHS (StockCode): ('m103', 'm119', 'm120')
RHS (StockCode): ('m118',)


In [21]:
# 最もリフト値が高いルール
max_lift_idx = df_res['Lift'].idxmax()
max_lift_rule = df_res.loc[max_lift_idx]

print(f"最もリフト値が高いルール:")
print(f"LHS (ID): {max_lift_rule['LHS']}")
print(f"RHS (ID): {max_lift_rule['RHS']}")
print(f"Confidence: {max_lift_rule['Conf']:.4f}")
print(f"Lift: {max_lift_rule['Lift']:.4f}")

# StockCodeを確認
print(f"\nLHS (StockCode): {tuple([id2sc[i] for i in max_lift_rule['LHS']])}")
print(f"RHS (StockCode): {tuple([id2sc[i] for i in max_lift_rule['RHS']])}")

最もリフト値が高いルール:
LHS (ID): (9, 12, 15)
RHS (ID): (18,)
Confidence: 0.8222
Lift: 1.4055

LHS (StockCode): ('m110', 'm113', 'm116')
RHS (StockCode): ('m119',)


In [22]:
display(df_res.sort_values(by='Lift', ascending=False))

,LHS,RHS,Conf,Lift
13,"(9, 12, 15)","(18,)",0.822222,1.405508
17,"(1, 14, 18)","(2,)",0.850000,1.393443
15,"(1, 14, 16)","(2,)",0.820513,1.345103
18,"(1, 14, 15)","(2,)",0.813953,1.334350
1,"(3, 17, 19)","(2,)",0.810811,1.329198
8,"(3, 5, 18)","(15,)",0.857143,1.328904
28,"(14, 15, 17)","(2,)",0.804348,1.318603
0,"(2, 18, 19)","(17,)",0.864865,1.310401
7,"(2, 4, 10)","(17,)",0.853659,1.293422
21,"(8, 12, 18)","(15,)",0.825000,1.279070
